# ChIP-seq bigWig enrichment in large genomic bins (mm10)

このノートブックでは、マウスゲノム mm10 を 0.5 Mb または 1 Mb のビンに区切り、
In this notebook, we partition the mouse mm10 genome into 0.5 Mb or 1 Mb bins,

bigWig 形式の ChIP-seq などのシグナルから各ビンの平均シグナル (enrichment) を計算します。
and compute the mean signal (enrichment) per bin from bigWig tracks (e.g., ChIP-seq).

`csv/bw-path.csv` に記載された複数の bigWig (ID, Path) をまとめて処理し、
We batch-process multiple bigWig files (ID, Path) listed in `csv/bw-path.csv`,

Nanog, Sox2, Usp5, Dnmt3l を含むビンがゲノムワイドな分布の中で高いか低いかを
and evaluate whether bins related to Nanog, Sox2, Usp5, and Dnmt3l are high or low relative to the genome-wide distribution,

z-score やパーセンタイルで評価します。
using z-scores and percentiles.

## 解析用 conda 環境メモ
Conda environment notes (analysis)

この Notebook では、ChIP-seq bigWig のビンごとの enrichment 計算のために、
This notebook uses a dedicated conda environment for computing per-bin enrichment from ChIP-seq bigWig files,
専用の conda 環境 `chipseq-enrich` を使用する。
named `chipseq-enrich`.

### 環境の作成手順 (macOS, zsh)
Environment setup (macOS, zsh)

```bash
# 新しい環境を作成
# Create a new environment
conda create -n chipseq-enrich python=3.10

# 環境を有効化
# Activate the environment
conda activate chipseq-enrich

# ライブラリのインストール
# Install libraries
conda install -c conda-forge numpy pandas matplotlib pybigwig

# Jupyter 関連のインストール
# Install Jupyter packages
conda install -c conda-forge jupyterlab ipykernel

# Jupyter カーネルとして登録
# Register as a Jupyter kernel
python -m ipykernel install --user \
  --name chipseq-enrich \
  --display-name "chipseq-enrich (ChIP-seq enrichment)"
```

In [ ]:
import numpy as np
import pandas as pd
import pyBigWig

print("NumPy:", np.__version__)
print("pandas:", pd.__version__)
print("pyBigWig:", pyBigWig.__version__)

In [ ]:
# 必要なライブラリのインポート
# Import required libraries
# 必要であれば、以下のセルを先に実行してインストールしてください:
# If needed, run the following first to install dependencies:
# %pip install pyBigWig pandas numpy matplotlib


import os
import math
import numpy as np
import pandas as pd
import pyBigWig
import matplotlib.pyplot as plt

plt.style.use("seaborn-v0_8")

# Input files are public bigWig tracks listed in bw-path.csv. Place them under
# GAO2026_EXTERNAL_ROOT/chipseq_enrichment, or override BW_TABLE_PATH directly.
from pathlib import Path

EXTERNAL_ROOT = Path(os.environ.get("GAO2026_EXTERNAL_ROOT", "external_inputs")).expanduser().resolve()
OUTPUT_ROOT = Path(os.environ.get("GAO2026_OUTPUT_ROOT", "outputs")).expanduser().resolve()
BW_TABLE_PATH = os.environ.get(
    "BW_TABLE_PATH",
    str(EXTERNAL_ROOT / "chipseq_enrichment" / "bw-path.csv"),
)

# ビンサイズ (Mb)。0.5 または 1 などに変更可能
# Bin size (Mb). You can set this to 0.5, 1, etc.
BIN_SIZE_MB = 0.5  # 0.5 などに変更可 / e.g., set to 0.5
BIN_SIZE_BP = int(BIN_SIZE_MB * 1_000_000) 

GENOME = "mm10"

## mm10染色体サイズと固定ビンの作成 / Load mm10 chrom sizes and create fixed bins

この次のセルでは、mm10 の染色体サイズを取得し、ゲノムワイドな固定ビン（例: 0.5 Mb または 1 Mb）を作成します。
In the next cell, we load mm10 chromosome sizes and create genome-wide fixed bins (e.g., 0.5 Mb or 1 Mb).

また、`csv/bw-path.csv` から bigWig の一覧（`ID`, `Path`）を読み込みます。
We also read the bigWig table (`ID`, `Path`) from `csv/bw-path.csv`.

In [ ]:
# mm10 の染色体サイズを取得し、ビンを作成する関数群
# Helper functions to load mm10 chromosome sizes and create genome bins


def load_mm10_chrom_sizes(from_url: bool = True,
                          url: str = "http://hgdownload.soe.ucsc.edu/goldenPath/mm10/bigZips/mm10.chrom.sizes") -> pd.DataFrame:
    """mm10 の染色体サイズを DataFrame として取得する。
    Load mm10 chromosome sizes as a DataFrame.

    デフォルトでは UCSC の mm10.chrom.sizes を HTTP で取得します。
    By default, this fetches UCSC `mm10.chrom.sizes` over HTTP.

    オフライン環境などで失敗する場合は、ローカルの chrom.sizes ファイルを
    読み込むようにコードを書き換えてください。
    If this fails (e.g., offline), modify the code to read a local `chrom.sizes` file instead.
    """

    if from_url:
        chrom_sizes = pd.read_csv(url, sep="\t", header=None, names=["chrom", "size"])
    else:
        raise ValueError(
            "from_url=False の場合は、ローカルファイル読み込み処理を実装してください。 "
            "(If from_url=False, please implement loading from a local file.)"
        )

    # 解析では基本的な常染色体と性染色体のみを利用
# Use autosomes and sex chromosomes only for analysis.
    chrom_keep = [f"chr{i}" for i in range(1, 20)] + ["chrX", "chrY"]
    chrom_sizes = chrom_sizes[chrom_sizes["chrom"].isin(chrom_keep)].reset_index(drop=True)
    chrom_sizes["size"] = chrom_sizes["size"].astype(int)
    return chrom_sizes



def make_genome_bins(chrom_sizes: pd.DataFrame, bin_size_bp: int) -> pd.DataFrame:
    """染色体サイズ情報からゲノムワイドなビンを作成する。
    Create genome-wide bins from chromosome size information.

    戻り値の DataFrame は以下のカラムを持つ:
    The returned DataFrame has the following columns:
      chrom, start, end, bin_id
    """

    records = []
    for _, row in chrom_sizes.iterrows():
        chrom = row["chrom"]
        size = int(row["size"])
        for start in range(0, size, bin_size_bp):
            end = min(start + bin_size_bp, size)
            bin_id = f"{chrom}:{start}-{end}"
            records.append({
                "chrom": chrom,
                "start": start,
                "end": end,
                "bin_id": bin_id,
            })

    bins = pd.DataFrame.from_records(records)
    return bins


def load_bw_table(csv_path: str) -> pd.DataFrame:
    """bigWig の ID とパスが書かれたテーブルを読み込む。
    Load a table containing bigWig IDs and paths.
    """
    bw_table = pd.read_csv(csv_path)
    if not {"ID", "Path"}.issubset(bw_table.columns):
        raise ValueError(
            "bw-path.csv には 'ID', 'Path' カラムが必要です。 "
            "(The file must contain 'ID' and 'Path' columns.)"
        )
    return bw_table


# 実際に mm10 染色体サイズとビンを作成
# Create mm10 chromosome sizes and genome-wide bins
chrom_sizes_mm10 = load_mm10_chrom_sizes()
bins = make_genome_bins(chrom_sizes_mm10, BIN_SIZE_BP)
print(f"Number of bins (bin size {BIN_SIZE_MB} Mb):", len(bins))

# bigWig テーブルの読み込み
# Load bigWig table
bw_table = load_bw_table(BW_TABLE_PATH)
bw_table

## 遺伝子TSSウィンドウの定義と enrichment 計算 / Define gene TSS windows and compute enrichment

この次のセルでは、興味のある遺伝子（例: Nanog, Sox2, Usp5, Dnmt3l）の座標と strand を設定し、TSS を推定します（`+` の場合は start、`-` の場合は end）。
In the next cell, we define gene coordinates and strands, and infer the TSS (`+` uses start, `-` uses end).

各 target（bigWig）について、(1) 固定ビンのゲノムワイド分布、(2) TSS中心ウィンドウの平均シグナルを計算し、z-score / percentile を算出して `results_df` を作成します。
For each target (bigWig), we compute (1) the genome-wide distribution on fixed bins and (2) mean signals in TSS-centered windows, then derive z-scores/percentiles and build `results_df`.

In [ ]:
# 遺伝子領域の定義と、bigWig からビンごとの enrichment を計算する関数
# Define gene regions and compute enrichment from bigWig per bin.
# ここで、興味のある遺伝子 (mm10) のゲノム座標を指定します。
# Specify the mm10 genomic coordinates of genes of interest here.
# それぞれ UCSC や Ensembl などで mm10 の座標を確認して、start/end を塩基番号で入力してください。
# Check mm10 coordinates (e.g., UCSC/Ensembl) and enter start/end as base positions.

genes_of_interest = [
    {"gene": "Nanog",  "chrom": "chr6",  "start": 122206999, "end": 123206999, "strand": "+"},
    {"gene": "Sox2",   "chrom": "chr3",  "start":  34149995, "end":  35149995, "strand": "+"},
    {"gene": "Usp5",   "chrom": "chr6",  "start": 124329395, "end": 125329395, "strand": "-"},
    {"gene": "Dnmt3L", "chrom": "chr10", "start":  77541947, "end":  78541947, "strand": "+"},
]


genes_df = pd.DataFrame(genes_of_interest)


# 座標が未入力のまま解析が進まないようにチェック
# Sanity-check to avoid running with missing coordinates
if genes_df[["chrom", "start", "end"]].isnull().any().any() or (genes_df["chrom"] == "chr?").any():
    raise ValueError(
        "genes_of_interest の chrom/start/end を mm10 に合わせて正しく入力してください。 "
        "(Please fill in chrom/start/end correctly for mm10 in genes_of_interest.)"
    )


def compute_bw_signal_per_bin(bw_path: str, bins_df: pd.DataFrame) -> pd.Series:
    """1 つの bigWig について、各ビンの平均シグナルを計算する。
    Compute mean signal per bin for a single bigWig.
    """

    if not os.path.exists(bw_path):
        raise FileNotFoundError(f"bigWig not found: {bw_path}")

    bw = pyBigWig.open(bw_path)
    values = []
    for _, row in bins_df.iterrows():
        v = bw.stats(row["chrom"], int(row["start"]), int(row["end"]), type="mean")[0]
        values.append(v)
    bw.close()
    return pd.Series(values, index=bins_df.index, name="signal")


def compute_bw_signal_for_intervals(
    bw_path: str,
    intervals_df: pd.DataFrame,
    chrom_col: str = "chrom",
    start_col: str = "start",
    end_col: str = "end",
    name: str = "signal",
) -> pd.Series:
    """任意の区間 (chrom/start/end) について、bigWig の平均シグナルを計算する。
    Compute mean bigWig signal for arbitrary intervals (chrom/start/end).
    """

    if not os.path.exists(bw_path):
        raise FileNotFoundError(f"bigWig not found: {bw_path}")

    bw = pyBigWig.open(bw_path)
    values = []
    for _, row in intervals_df.iterrows():
        chrom = row[chrom_col]
        start = int(row[start_col])
        end = int(row[end_col])
        v = bw.stats(chrom, start, end, type="mean")[0]
        values.append(v)
    bw.close()
    return pd.Series(values, index=intervals_df.index, name=name)


def resolve_tss_from_strand(genes_df: pd.DataFrame) -> pd.DataFrame:
    """strand と start/end から TSS を推定して付与する。
    Infer and add TSS from strand and start/end.

    strand が '+' の場合は start、'-' の場合は end を TSS とする。
    If strand is '+', use start as TSS; if '-', use end as TSS.
    """

    required = {"gene", "chrom", "start", "end", "strand"}
    missing = required - set(genes_df.columns)
    if missing:
        raise ValueError(
            f"genes_df に必要なカラムがありません: {sorted(missing)} "
            f"(Missing required columns in genes_df: {sorted(missing)})"
        )

    out = genes_df.copy()
    out["strand"] = out["strand"].astype(str)
    invalid = ~out["strand"].isin(["+", "-"])
    if invalid.any():
        bad = out.loc[invalid, ["gene", "strand"]].to_dict(orient="records")
        raise ValueError(
            f"strand は '+' か '-' を指定してください。問題の行: {bad} "
            f"(strand must be '+' or '-'. Problematic rows: {bad})"
        )

    out["start"] = out["start"].astype(int)
    out["end"] = out["end"].astype(int)
    out["tss"] = np.where(out["strand"] == "+", out["start"], out["end"]).astype(int)
    out["tss_source"] = "strand"
    return out


def make_tss_windows(
    genes_df_with_tss: pd.DataFrame,
    chrom_sizes_df: pd.DataFrame,
    window_size_bp: int,
) -> pd.DataFrame:
    """各遺伝子について、TSS を中心に window_size_bp のウィンドウを作る。
    Create a window of size window_size_bp centered on TSS for each gene.

    染色体端にかかる場合は、長さを保ったまま染色体内に収まるようにシフトする。
    If the window crosses chromosome ends, shift it to keep the length while staying within the chromosome.
    """

    chrom_size_map = dict(zip(chrom_sizes_df["chrom"], chrom_sizes_df["size"]))
    half = int(window_size_bp // 2)

    records = []
    for _, g in genes_df_with_tss.iterrows():
        chrom = g["chrom"]
        if chrom not in chrom_size_map:
            raise ValueError(
                f"chrom_sizes に存在しない染色体です: {chrom} "
                f"(Chromosome not found in chrom_sizes: {chrom})"
            )
        chrom_size = int(chrom_size_map[chrom])

        tss = int(g["tss"])
        window_start = tss - half
        window_end = window_start + int(window_size_bp)

        # クリップ + シフトで染色体内に収める (長さは維持)
        # Clip + shift to stay within chromosome while preserving length
        if window_start < 0:
            window_start = 0
            window_end = int(window_size_bp)
        if window_end > chrom_size:
            window_end = chrom_size
            window_start = max(0, window_end - int(window_size_bp))

        records.append({
            "gene": g["gene"],
            "chrom": chrom,
            "gene_start": int(g["start"]),
            "gene_end": int(g["end"]),
            "strand": g["strand"],
            "tss": tss,
            "tss_source": g.get("tss_source", "strand"),
            "window_start": int(window_start),
            "window_end": int(window_end),
        })

    return pd.DataFrame.from_records(records)


genes_df_tss = resolve_tss_from_strand(genes_df)
gene_windows = make_tss_windows(genes_df_tss, chrom_sizes_mm10, BIN_SIZE_BP)


results = []

for _, bw_row in bw_table.iterrows():
    target_id = bw_row["ID"]
    bw_path = bw_row["Path"]
    print(f"Processing: {target_id} ({bw_path})")

    # ゲノムワイド分布は固定ビンで計算 (従来どおり)
    # Compute genome-wide background distribution using fixed bins (as before)
    signal = compute_bw_signal_per_bin(bw_path, bins)
    bins_with_signal = bins.copy()
    bins_with_signal["signal"] = signal

    genome_vals = bins_with_signal["signal"].astype(float).dropna().values
    if len(genome_vals) == 0:
        print(f"  Warning: no signal values for {target_id}")
        continue

    genome_mean = float(np.mean(genome_vals))
    genome_std = float(np.std(genome_vals))

    # 遺伝子ごとは、TSS 中心ウィンドウで mean を直接計算
    # For each gene, compute mean signal directly in the TSS-centered window
    gene_signal = compute_bw_signal_for_intervals(
        bw_path,
        gene_windows,
        chrom_col="chrom",
        start_col="window_start",
        end_col="window_end",
        name="signal",
    )

    for idx, gw in gene_windows.iterrows():
        val = float(gene_signal.loc[idx]) if not pd.isna(gene_signal.loc[idx]) else np.nan
        if math.isnan(val):
            continue

        percentile = float((genome_vals < val).mean() * 100.0)
        z = (val - genome_mean) / genome_std if genome_std > 0 else np.nan

        results.append({
            "target": target_id,
            "bw_path": bw_path,
            "bin_size_bp": BIN_SIZE_BP,
            "window_size_bp": BIN_SIZE_BP,
            "gene": gw["gene"],
            "chrom": gw["chrom"],
            "strand": gw["strand"],
            "tss": int(gw["tss"]),
            "tss_source": gw["tss_source"],
            # 互換性のために bin_start/bin_end も残す (TSS ウィンドウを指す)
            # Keep bin_start/bin_end for compatibility (they refer to the TSS window)
            "bin_start": int(gw["window_start"]),
            "bin_end": int(gw["window_end"]),
            "window_start": int(gw["window_start"]),
            "window_end": int(gw["window_end"]),
            "gene_start": int(gw["gene_start"]),
            "gene_end": int(gw["gene_end"]),
            "signal": val,
            "genome_mean": genome_mean,
            "genome_std": genome_std,
            "z_score": z,
            "percentile": percentile,
        })

results_df = pd.DataFrame(results)
print("Number of gene-window measurements:", len(results_df))
results_df

## 結果の簡易チェック（例） / Quick sanity-check plot (example)

この次のセルでは、計算結果の確認として、1つの target を例にゲノムワイド分布（固定ビン）と遺伝子TSSウィンドウの値を同じ図に表示します。
In the next cell, as a sanity check, we plot one example target: the genome-wide fixed-bin distribution and the gene TSS-window values on the same figure.

In [ ]:
# 結果の確認と簡単な可視化
# Quick result check and a simple visualization


if not results_df.empty:
    display(results_df.head())

    # 例として、特定の標的についてゲノムワイド分布と TSS 中心ウィンドウを可視化
    # Example: visualize genome-wide distribution and TSS-centered windows for one target
    target_to_plot = results_df["target"].unique()[0]
    print("Example plot for target:", target_to_plot)

    # 対象 bigWig のゲノムワイド分布を再計算 (固定ビン)
    # Recompute the genome-wide distribution for the target bigWig (fixed bins)
    bw_row = bw_table[bw_table["ID"] == target_to_plot].iloc[0]
    bw_path = bw_row["Path"]
    signal = compute_bw_signal_per_bin(bw_path, bins)
    genome_vals = signal.astype(float).dropna().values

    # TSS 中心ウィンドウでの値 (gene ごとに 1 つ)
    # Values in TSS-centered windows (one per gene)
    gene_vals = results_df.loc[results_df["target"] == target_to_plot, "signal"].values

    plt.figure(figsize=(6, 4))
    plt.hist(genome_vals, bins=50, alpha=0.7, label="Genome-wide fixed bins")
    for gv in gene_vals:
        plt.axvline(gv, color="red", linestyle="--", linewidth=2)
    plt.xlabel("Mean signal per window")
    plt.ylabel("Count")
    plt.title(f"{target_to_plot}: genome-wide bins vs TSS-centered windows")
    plt.legend()
    plt.tight_layout()
    plt.show()

else:
    print("results_df is empty. Check gene coordinates and bigWig inputs.")

## `results_df` の表示 / Display `results_df`

この次のセルでは、最終結果テーブル `results_df` をそのまま表示します。
In the next cell, we display the final results table `results_df`.

In [ ]:
results_df

## Target別のゲノムワイド分布（固定ビン）と、遺伝子TSSウィンドウの位置を同じ図で表示
## Per-target genome-wide distribution (fixed bins) with gene TSS-window values overlaid

このセルでは以下を可視化します。
This cell visualizes the following:

- **箱ひげ図**: 各 `target` について、固定ビン（`bins`）で計算した *mean bigWig signal* の分布（ゲノムワイド）
- **Boxplot**: For each `target`, the genome-wide distribution of *mean bigWig signal* computed on fixed bins (`bins`)
- **点（色分け）**: `results_df` にある各 `gene` の *mean bigWig signal*（TSS中心ウィンドウ）を、同じ `target` 上に重ねて表示
- **Colored points**: For each `gene` in `results_df`, the *mean bigWig signal* in the TSS-centered window, overlaid on the same `target`

※ `bw_table` に同じ `ID` が複数ある場合でも、ここでは **targetごとに先頭の `bw_path` を代表として箱ひげ図**を作成します（replicateで重み付けしないため）。
Note: If `bw_table` contains multiple rows with the same `ID`, we still build the boxplot using the **first `bw_path` per target** (i.e., no replicate weighting).

In [ ]:
# Target別 boxplot（固定ビンのゲノムワイド分布） + gene点（TSS中心ウィンドウ）
# Per-target boxplot (genome-wide fixed-bin distribution) + gene points (TSS-centered windows)

# 1) 描画順（target）を results_df の出現順で固定
# 1) Fix plotting order (targets) by appearance order in results_df
targets = list(pd.unique(results_df["target"]))

# 2) 箱ひげ図用：targetごとに代表bw_pathを1つ選び、固定ビンの分布を再計算
# 2) For boxplots: pick one representative bw_path per target and recompute fixed-bin distributions
bw_table_unique = bw_table.drop_duplicates(subset=["ID"], keep="first")
bw_table_unique = bw_table_unique[bw_table_unique["ID"].isin(targets)]

genome_bin_records = []
for _, row in bw_table_unique.iterrows():
    target = row["ID"]
    bw_path = row["Path"]
    s = compute_bw_signal_per_bin(bw_path, bins).astype(float)
    s = s.dropna()
    # binsのbin_idと紐づけて記録
    # Record together with the corresponding bin_id from bins
    for bin_id, val in zip(bins.loc[s.index, "bin_id"].values, s.values):
        genome_bin_records.append({
            "target": target,
            "bw_path": bw_path,
            "bin_id": bin_id,
            "mean_bigwig_signal": float(val),
        })

genome_bins_df = pd.DataFrame(genome_bin_records)

# 3) gene点用：replicateがあっても平均で1点にする（target×geneで平均）
# 3) For gene points: average to one point per target×gene (even if replicates exist)
gene_points_df = (
    results_df[["target", "gene", "signal"]]
    .dropna(subset=["signal"])
    .groupby(["target", "gene"], as_index=False)["signal"]
    .mean()
    .rename(columns={"signal": "mean_bigwig_signal"})
)

genes = list(pd.unique(gene_points_df["gene"]))
n_genes = max(1, len(genes))

# 4) matplotlibで描画
# 4) Plot with matplotlib
fig, ax = plt.subplots(figsize=(max(8, 0.6 * len(targets)), 5))

# boxplotデータ（target順）
# Boxplot data (in target order)
box_data = [
    genome_bins_df.loc[genome_bins_df["target"] == t, "mean_bigwig_signal"].values
    for t in targets
 ]

bp = ax.boxplot(
    box_data,
    positions=np.arange(len(targets)),
    widths=0.6,
    showfliers=False,
    patch_artist=True,
)
for patch in bp["boxes"]:
    patch.set_alpha(0.35)

# gene点を色分けして重ねる（target位置に少しだけオフセット）
# Overlay gene points with colors (slight x-offset at each target position)
cmap = plt.get_cmap("tab10")
gene_to_color = {g: cmap(i % 10) for i, g in enumerate(genes)}

for gi, g in enumerate(genes):
    sub = gene_points_df[gene_points_df["gene"] == g]
    x_base = np.array([targets.index(t) for t in sub["target"].values], dtype=float)
    offset = (gi - (n_genes - 1) / 2) * 0.10
    x = x_base + offset
    y = sub["mean_bigwig_signal"].values
    ax.scatter(x, y, s=50, color=gene_to_color[g], edgecolor="black", linewidth=0.4, label=g, zorder=3)

ax.set_xticks(np.arange(len(targets)))
ax.set_xticklabels(targets, rotation=45, ha="right")
ax.set_xlabel("target")
ax.set_ylabel("Mean bigWig signal")
ax.set_title("Genome-wide fixed-bin distribution (boxplot) with gene TSS-window signals (points)")
ax.legend(title="gene", bbox_to_anchor=(1.02, 1), loc="upper left", borderaxespad=0)
plt.tight_layout()
plt.show()

## Target別 joint z-score（固定ビン＋gene値を含めて標準化）
## Per-target joint z-score (standardize using fixed bins + gene values)

target間で mean bigWig signal のスケール差が大きい場合、同じy軸で比較すると見にくくなります。
When mean bigWig signal scales differ substantially across targets, a shared y-axis can be hard to interpret.
そこでこのセルでは、各 target について以下を同じ母集団として扱い、z-score（標準化）します。
In this cell, for each target we treat the following as one population and compute a standardized z-score:

- 固定ビン（`bins`）で計算したゲノムワイドの mean bigWig signal 分布
- Genome-wide mean bigWig signal distribution computed on fixed bins (`bins`)
- `results_df` の各 `gene` の mean bigWig signal（TSS中心ウィンドウ）
- Mean bigWig signal for each `gene` in `results_df` (TSS-centered window)

各 target ごとに、上記を結合した値から平均 $\mu_t$ と標準偏差 $\sigma_t$ を計算し、
For each target, we compute the mean $\mu_t$ and standard deviation $\sigma_t$ from the combined values above,
$z = (x - \mu_t) / \sigma_t$ として z-score に変換します。
and convert to z-scores as $z = (x - \mu_t) / \sigma_t$.

可視化：
Visualization:
- **箱ひげ図**：固定ビン（ゲノムワイド）由来の z-score 分布（target別）
- **Boxplot**: z-score distribution from genome-wide fixed bins (per target)
- **点（色分け）**：gene（TSS中心ウィンドウ）の z-score を重ね描き
- **Colored points**: overlaid z-scores for gene (TSS-centered window) values

In [ ]:
# Target別 joint z-score（fixed bins + gene windows を同じ母集団として標準化）
# Per-target joint z-score (standardize using fixed bins + gene windows as one population)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# 1) 描画順（target）を results_df の出現順で固定
# 1) Fix plotting order (targets) by appearance order in results_df
targets_joint = list(pd.unique(results_df["target"]))

# 2) 箱ひげ図用：targetごとに代表bw_pathを1つ選び、固定ビン分布を再計算
# 2) For boxplots: pick one representative bw_path per target and recompute fixed-bin distributions
bw_table_unique_joint = bw_table.drop_duplicates(subset=["ID"], keep="first")
bw_table_unique_joint = bw_table_unique_joint[bw_table_unique_joint["ID"].isin(targets_joint)].copy()

# 3) gene点用：replicateがあっても target×gene で平均して 1点にする
# 3) For gene points: average to one point per target×gene (even if replicates exist)
gene_points_joint = (
    results_df[["target", "gene", "signal"]]
    .dropna(subset=["signal"])
    .groupby(["target", "gene"], as_index=False)["signal"]
    .mean()
    .rename(columns={"signal": "gene_signal"})
)

genome_z_records = []
gene_z_records = []
warnings = []

for _, row in bw_table_unique_joint.iterrows():
    target = row["ID"]
    bw_path = row["Path"]

    genome_vals = compute_bw_signal_per_bin(bw_path, bins).astype(float).dropna().values
    genome_vals = genome_vals[np.isfinite(genome_vals)]
    if genome_vals.size == 0:
        warnings.append(f"No genome-wide bin values for target={target}")
        continue

    gene_sub = gene_points_joint.loc[gene_points_joint["target"] == target].copy()
    gene_vals = gene_sub["gene_signal"].astype(float).values
    gene_vals = gene_vals[np.isfinite(gene_vals)]

    combined = np.concatenate([genome_vals, gene_vals]) if gene_vals.size else genome_vals
    combined = combined[np.isfinite(combined)]
    if combined.size == 0:
        warnings.append(f"No combined values for target={target}")
        continue

    mu = float(np.mean(combined))
    sigma = float(np.std(combined))  # ddof=0（ノートブック内のstd計算と整合） / ddof=0 (consistent with std calculation elsewhere in this notebook)
    if (not np.isfinite(sigma)) or sigma <= 0:
        warnings.append(f"sigma<=0 for target={target}; z-score skipped")
        continue

    z_genome = (genome_vals - mu) / sigma
    z_genome = z_genome[np.isfinite(z_genome)]
    genome_z_records.extend({"target": target, "z_joint": float(z)} for z in z_genome)

    if not gene_sub.empty:
        gene_sub["z_joint"] = (gene_sub["gene_signal"] - mu) / sigma
        gene_sub = gene_sub.dropna(subset=["z_joint"])
        gene_z_records.extend(
            gene_sub[["target", "gene", "z_joint"]].to_dict(orient="records")
        )

genome_z_df = pd.DataFrame(genome_z_records)
gene_z_df = pd.DataFrame(gene_z_records)

# 4) Plot: boxplot (genome bins) + overlay gene points (colored by gene)
targets_plot = [t for t in targets_joint if t in set(genome_z_df["target"]) ]
genes = list(pd.unique(gene_z_df["gene"])) if not gene_z_df.empty else []
n_genes = max(1, len(genes))

fig, ax = plt.subplots(figsize=(max(8, 0.6 * len(targets_plot)), 5))

box_data = [
    genome_z_df.loc[genome_z_df["target"] == t, "z_joint"].values
    for t in targets_plot
 ]

ax.boxplot(
    box_data,
    positions=np.arange(len(targets_plot)),
    widths=0.6,
    showfliers=False,
)

# gene points (slight x-offset per gene)
cmap = plt.get_cmap("tab10")
gene_to_color = {g: cmap(i % 10) for i, g in enumerate(genes)}
labeled = set()

for gi, g in enumerate(genes):
    sub = gene_z_df[gene_z_df["gene"] == g]
    if sub.empty:
        continue
    x_base = np.array([targets_plot.index(t) for t in sub["target"].values], dtype=float)
    offset = (gi - (n_genes - 1) / 2) * 0.10
    x = x_base + offset
    y = sub["z_joint"].values
    label = g if g not in labeled else None
    ax.scatter(
        x, y,
        s=50,
        color=gene_to_color[g],
        edgecolor="black",
        linewidth=0.4,
        label=label,
        zorder=3,
    )
    labeled.add(g)

ax.set_xticks(np.arange(len(targets_plot)))
ax.set_xticklabels(targets_plot, rotation=45, ha="right")
ax.set_xlabel("target")
ax.set_ylabel("Joint z-score")
ax.set_title("Per-target joint z-scores (genome-wide fixed bins + gene windows)")
if genes:
    ax.legend(title="gene", bbox_to_anchor=(1.02, 1), loc="upper left", borderaxespad=0)
plt.tight_layout()
plt.show()

if warnings:
    print("Warnings:")
    for w in warnings:
        print("-", w)

In [ ]:
# Target別 joint z-score（fixed bins + gene windows を同じ母集団として標準化）
# Per-target joint z-score (standardize using fixed bins + gene windows as one population)

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# --- Styling (cell-local) ---
BASE_FONTSIZE = 12
FONT_FAMILY = "Arial"  # fallback will be used if unavailable

rc = {
    "font.family": "sans-serif",
    "font.sans-serif": [FONT_FAMILY, "DejaVu Sans"],
    "font.size": BASE_FONTSIZE,
    "axes.titlesize": BASE_FONTSIZE,
    "axes.labelsize": BASE_FONTSIZE,
    "xtick.labelsize": BASE_FONTSIZE,
    "ytick.labelsize": BASE_FONTSIZE,
    "legend.fontsize": BASE_FONTSIZE,
    "legend.title_fontsize": BASE_FONTSIZE,
}

with plt.rc_context(rc):
    # 1) 描画順（target）を results_df の出現順で固定
    # 1) Fix plotting order (targets) by appearance order in results_df
    targets_joint = list(pd.unique(results_df["target"]))

    # 2) 箱ひげ図用：targetごとに代表bw_pathを1つ選び、固定ビン分布を再計算
    # 2) For boxplots: pick one representative bw_path per target and recompute fixed-bin distributions
    bw_table_unique_joint = bw_table.drop_duplicates(subset=["ID"], keep="first")
    bw_table_unique_joint = bw_table_unique_joint[bw_table_unique_joint["ID"].isin(targets_joint)].copy()

    # 3) gene点用：replicateがあっても target×gene で平均して 1点にする
    # 3) For gene points: average to one point per target×gene (even if replicates exist)
    gene_points_joint = (
        results_df[["target", "gene", "signal"]]
        .dropna(subset=["signal"])
        .groupby(["target", "gene"], as_index=False)["signal"]
        .mean()
        .rename(columns={"signal": "gene_signal"})
    )

    genome_z_records = []
    gene_z_records = []
    warnings = []

    for _, row in bw_table_unique_joint.iterrows():
        target = row["ID"]
        bw_path = row["Path"]

        genome_vals = compute_bw_signal_per_bin(bw_path, bins).astype(float).dropna().values
        genome_vals = genome_vals[np.isfinite(genome_vals)]
        if genome_vals.size == 0:
            warnings.append(f"No genome-wide bin values for target={target}")
            continue

        gene_sub = gene_points_joint.loc[gene_points_joint["target"] == target].copy()
        gene_vals = gene_sub["gene_signal"].astype(float).values
        gene_vals = gene_vals[np.isfinite(gene_vals)]

        combined = np.concatenate([genome_vals, gene_vals]) if gene_vals.size else genome_vals
        combined = combined[np.isfinite(combined)]
        if combined.size == 0:
            warnings.append(f"No combined values for target={target}")
            continue

        mu = float(np.mean(combined))
        sigma = float(np.std(combined))  # ddof=0（ノートブック内のstd計算と整合） / ddof=0 (consistent with std calculation elsewhere in this notebook)
        if (not np.isfinite(sigma)) or sigma <= 0:
            warnings.append(f"sigma<=0 for target={target}; z-score skipped")
            continue

        z_genome = (genome_vals - mu) / sigma
        z_genome = z_genome[np.isfinite(z_genome)]
        genome_z_records.extend({"target": target, "z_joint": float(z)} for z in z_genome)

        if not gene_sub.empty:
            gene_sub["z_joint"] = (gene_sub["gene_signal"] - mu) / sigma
            gene_sub = gene_sub.dropna(subset=["z_joint"])
            gene_z_records.extend(
                gene_sub[["target", "gene", "z_joint"]].to_dict(orient="records"),
            )

    genome_z_df = pd.DataFrame(genome_z_records)
    gene_z_df = pd.DataFrame(gene_z_records)

    # 4) Plot: boxplot (genome bins) + overlay gene points (colored by gene)
    targets_plot = [t for t in targets_joint if t in set(genome_z_df["target"]) ]
    genes = list(pd.unique(gene_z_df["gene"])) if not gene_z_df.empty else []
    n_genes = max(1, len(genes))

    fig, ax = plt.subplots(figsize=(max(8, 0.6 * len(targets_plot)), 4))

    box_data = [
        genome_z_df.loc[genome_z_df["target"] == t, "z_joint"].values
        for t in targets_plot
    ]

    # White-filled boxes
    ax.boxplot(
        box_data,
        positions=np.arange(len(targets_plot)),
        widths=0.6,
        showfliers=False,
        patch_artist=True,
        boxprops=dict(facecolor="white", edgecolor="black", linewidth=1.0),
        medianprops=dict(color="black", linewidth=1.0),
        whiskerprops=dict(color="black", linewidth=1.0),
        capprops=dict(color="black", linewidth=1.0),
    )

    # gene points (slight x-offset per gene)
    cmap = plt.get_cmap("tab10")
    gene_to_color = {g: cmap(i % 10) for i, g in enumerate(genes)}
    labeled = set()

    for gi, g in enumerate(genes):
        sub = gene_z_df[gene_z_df["gene"] == g]
        if sub.empty:
            continue
        x_base = np.array([targets_plot.index(t) for t in sub["target"].values], dtype=float)
        offset = (gi - (n_genes - 1) / 2) * 0.10
        x = x_base + offset
        y = sub["z_joint"].values
        label = g if g not in labeled else None
        ax.scatter(
            x, y,
            s=50,
            color=gene_to_color[g],
            edgecolor="black",
            linewidth=0.4,
            label=label,
            zorder=3,
        )
        labeled.add(g)

    ax.set_xticks(np.arange(len(targets_plot)))
    ax.set_xticklabels(targets_plot, rotation=45, ha="right")
    ax.set_xlabel("target")
    ax.set_ylabel("Joint z-score")
    ax.set_title("Per-target joint z-scores (genome-wide fixed bins + gene windows)")

    if genes:
        leg = ax.legend(title="gene", bbox_to_anchor=(1.02, 1), loc="upper left", borderaxespad=0)
        for t in leg.get_texts():
            t.set_fontstyle("italic")

    plt.tight_layout()

    # Save PDF and still show inline
    figure_output_root = OUTPUT_ROOT / "15_chipseq_enrichment"
    figure_output_root.mkdir(parents=True, exist_ok=True)
    out_pdf = figure_output_root / "joint_zscore_per_target_fixedbins_genewindows.pdf"
    fig.savefig(out_pdf, format="pdf", bbox_inches="tight")
    plt.show()

    if warnings:
        print("Warnings:")
        for w in warnings:
            print("-", w)